# 1) Install & Imports

In [2]:
!pip install pypdf sentence-transformers faiss-cpu numpy


In [3]:
!pip install "pydantic>=2.6" "pydantic-core>=2.16"
!pip install -q pypdf sentence-transformers faiss-cpu numpy pandas requests


In [4]:
import os, glob, re
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss


# 2) Config

In [5]:
import os

# --- Keys / endpoints ---
GEMINI_KEY = os.getenv("GEMINI_API_KEY")  # set this in your OS env
if not GEMINI_KEY:
    raise ValueError("Missing GEMINI_API_KEY environment variable.")

GEMINI_MODEL = "gemini-2.5-flash"   # or "gemini-2.5-flash-lite"
GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_KEY}"

OLLAMA_URL = "http://localhost:11434/api/generate"


In [6]:
USE_GEMINI = True  # set False if you only want local runs
GEMINI_MAX_CALLS = 6       # total Gemini calls allowed in a run (small budget)

# 3) Load PDFs

In [7]:
DATA_DIR = r"C:\Users\Adan\OneDrive - University of Haifa\אחזור מידע\final_assignment\RAG_Dinosaurs_Data"
PDF_PATHS = glob.glob(os.path.join(DATA_DIR, "*.pdf"))
len(PDF_PATHS), PDF_PATHS[:3]


(15,
 ['C:\\Users\\Adan\\OneDrive - University of Haifa\\אחזור מידע\\final_assignment\\RAG_Dinosaurs_Data\\Allosaurus - Wikipedia.pdf',
  'C:\\Users\\Adan\\OneDrive - University of Haifa\\אחזור מידע\\final_assignment\\RAG_Dinosaurs_Data\\Ankylosaurus - Wikipedia.pdf',
  'C:\\Users\\Adan\\OneDrive - University of Haifa\\אחזור מידע\\final_assignment\\RAG_Dinosaurs_Data\\Brachiosaurus - Wikipedia.pdf'])

In [8]:
def read_pdf_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = []
    for p in reader.pages:
        t = p.extract_text() or ""
        pages.append(t)
    text = "\n".join(pages)
    text = re.sub(r"\s+", " ", text).strip()
    return text

docs = []
for path in PDF_PATHS:
    docs.append({
        "source": os.path.basename(path),
        "text": read_pdf_text(path)
    })

len(docs), docs[0]["source"], docs[0]["text"][:500]


(15,
 'Allosaurus - Wikipedia.pdf',
 'Allosaurus Temporal range: Late Jurassic (Kimmeridgian to Tithonian), Mounted cast of the specimen "Big Al 2" (Allosaurus jimmadseni) during a special exhibit at the Museum Koenig Bonn Scientific classification Kingdom: Animalia Phylum: Chordata Class: Reptilia Clade: Dinosauria Clade: Saurischia Clade: Theropoda Family: †Allosauridae Subfamily: †Allosaurinae Marsh, 1878 Genus: †Allosaurus Marsh, 1877 Type species †Allosaurus fragilis Marsh, 1877 Other species †A. europaeus Mateus et al., 2006 A')

# 4) Chunking

In [9]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 120):
    chunks = []
    i = 0
    while i < len(text):
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)
        i += (chunk_size - overlap)
    return chunks

def build_chunks(docs, chunk_size=800, overlap=120):
    all_chunks = []
    for d in docs:
        for c in chunk_text(d["text"], chunk_size=chunk_size, overlap=overlap):
            all_chunks.append({
                "source": d["source"],
                "chunk": c
            })
    return all_chunks

chunks = build_chunks(docs, chunk_size=800, overlap=120)
len(chunks)


2143

# 5) Embeddings

In [10]:
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMB_MODEL_NAME)

def embed_texts(texts):
    vecs = embedder.encode(
        texts,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return vecs.astype("float32")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Create vectors ONCE from chunks
chunk_texts = [c["chunk"] for c in chunks]
vectors = embed_texts(chunk_texts)
print(vectors.shape)


(2143, 384)


# 6) Indexing (Flat / HNSW / IVF)

In [13]:
# --- Different FAISS index builders ---
import faiss
import numpy as np

def build_faiss_flat_index(vectors: np.ndarray):
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine-like if vectors are normalized
    index.add(vectors)
    return index

def build_faiss_hnsw_index(vectors: np.ndarray, M: int = 32, ef_construction: int = 200):
    dim = vectors.shape[1]
    index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = ef_construction
    index.add(vectors)
    return index

def build_faiss_ivf_index(vectors: np.ndarray, nlist: int = 16, nprobe: int = 8):
    """
    IVF needs training. nlist = number of clusters.
    nprobe controls recall/speed tradeoff at query time.
    """
    dim = vectors.shape[1]
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)

    # train on the vectors (needs enough points; if small dataset, use smaller nlist)
    index.train(vectors)
    index.add(vectors)
    index.nprobe = nprobe
    return index


In [14]:
# --- Build all indexes ---
index_flat = build_faiss_flat_index(vectors)
index_hnsw = build_faiss_hnsw_index(vectors)
index_ivf  = build_faiss_ivf_index(vectors)

INDEXES = {
    "flat": index_flat,
    "hnsw": index_hnsw,
    "ivf":  index_ivf,
}


In [15]:
test_vec = embedder.encode(["hello"], convert_to_numpy=True, normalize_embeddings=True)
test_vec.shape


(1, 384)

# 7) Retrieval

In [18]:
def retrieve(query, chunks, index, top_k=5):
    # 1) Embed the query
    q_vec = embed_texts([query])  # shape (1, dim)

    # 2) Search in the given index (Flat / HNSW / IVF — doesn't matter)
    scores, indices = index.search(q_vec, top_k)

    # 3) Collect results
    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        results.append({
            "rank": rank + 1,
            "score": float(scores[0][rank]),
            "chunk": chunks[idx]["chunk"],
            "source": chunks[idx]["source"],
        })

    return results



In [19]:
query = "Which dinosaur is known for having a sail on its back?"

# choose the index algorithm here:
results = retrieve(query, chunks, INDEXES["hnsw"], top_k=5)
# try "flat" or "ivf" too

for r in results:
    print(f"\n#{r['rank']} score={r['score']:.3f} source={r['source']}")
    print(r["chunk"])



#1 score=0.708 source=Spinosaurus - Wikipedia.pdf
ugh other dinosaurs, namely Ouranosaurus, which lived a few million years earlier in the same general region as Spinosaurus, and the Early Cretaceous South American sauropod Amargasaurus, might have developed similar structural adaptations of their vertebrae. The sail may be an analog of the sail of the Permian synapsid Dimetrodon, which lived before the dinosaurs even appeared, produced by convergent evolution.[92] Skull Postcranial skeleton 1/31/26, 2:59 PM Spinosaurus - Wikipedia https://en.wikipedia.org/wiki/Spinosaurus 15/48 Life restoration Restoration of various spinosaurids that did not live in the same time or space; Spinosaurus, Baryonyx, Irritator and Suchomimus The structure may also have been more hump-like than sail-like, as noted by Stromer in 1915 ("one might rather think o

#2 score=0.703 source=Spinosaurus - Wikipedia.pdf
d claws, suggesting that its hands were longer compared to those of other spinosaurids.[15][59][3

# 8) Prompt

In [24]:
def build_rag_prompt(query, retrieved):
    context = "\n\n".join([f"[{r['source']}] {r['chunk'][:400]}" for r in retrieved])
    return f"""Answer ONLY using the context.

QUESTION:
{query}

CONTEXT:
{context}

ANSWER:
"""
    return prompt

def rag(query, chunks, index, top_k=5):
    retrieved = retrieve(query, chunks, index, top_k=top_k)
    prompt = build_rag_prompt(query, retrieved)
    return prompt, retrieved


In [25]:
query = "Which dinosaur is known for having a sail on its back?"
prompt, retrieved = rag(query, chunks, INDEXES["hnsw"], top_k=5)

print(prompt[:1500])   # תראי שהקונטקסט נכנס יפה



Answer ONLY using the context.

QUESTION:
Which dinosaur is known for having a sail on its back?

CONTEXT:
[Spinosaurus - Wikipedia.pdf] ugh other dinosaurs, namely Ouranosaurus, which lived a few million years earlier in the same general region as Spinosaurus, and the Early Cretaceous South American sauropod Amargasaurus, might have developed similar structural adaptations of their vertebrae. The sail may be an analog of the sail of the Permian synapsid Dimetrodon, which lived before the dinosaurs even appeared, produced by conver

[Spinosaurus - Wikipedia.pdf] d claws, suggesting that its hands were longer compared to those of other spinosaurids.[15][59][35] Very tall neural spines growing on the back vertebrae of Spinosaurus formed the basis of what is usually called the animal's "sail". The lengths of the neural spines reached over 10 times the diameters of the centra (vertebral bodies) from which they extended.[92][93] The neural spines were slightly

[Spinosaurus - Wikipedia.pdf]

# 9) LLM wrappers (Ollama / Gemini)

In [26]:
import requests

import requests

import requests

import requests

def ollama_generate(prompt: str, model: str = "llama3:latest"):
    payload = {"model": model, "prompt": prompt, "stream": False}

    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=180)
    except Exception as e:
        raise RuntimeError(f"Request to Ollama failed before response: {repr(e)}")

    print("STATUS:", r.status_code)
    print("RAW TEXT (first 1000 chars):")
    print(r.text[:1000])

    if not r.ok:
        raise RuntimeError("Ollama failed (see printed STATUS / RAW TEXT above).")

    data = r.json()

    if "error" in data:
        raise RuntimeError(f"Ollama JSON error: {data['error']}")

    return data.get("response", "")



def rag_answer(query, chunks, index, top_k=5, model="llama3"):
    retrieved = retrieve(query, chunks, index, top_k=top_k)
    prompt = build_rag_prompt(query, retrieved)
    answer = ollama_generate(prompt, model=model)
    return answer, retrieved


In [27]:
q = "Which dinosaur is known for having a sail on its back?"
ans, src = rag_answer(q, chunks, INDEXES["hnsw"], top_k=2, model="mistral:latest")

print("ANSWER:\n", ans)
print("\nTOP SOURCES:")
for r in src[:3]:
    print(f"- {r['source']} (score={r['score']:.3f})")


STATUS: 200
RAW TEXT (first 1000 chars):
{"model":"mistral:latest","created_at":"2026-02-01T21:07:02.7924754Z","response":" Spinosaurus","done":true,"done_reason":"stop","context":[3,29473,27075,10456,10648,2181,1040,3526,29491,781,781,29592,12939,2470,29515,781,27471,7534,1153,4275,1117,3419,1122,3229,1032,13200,1124,1639,1620,29572,781,781,27949,29515,781,29560,5709,8673,25076,1155,22508,29491,9752,29561,1100,1359,1567,7534,6457,2494,29493,20670,4257,20581,25076,29493,1458,7030,1032,2432,4609,2035,6353,1065,1040,2116,3720,5192,1158,2438,8673,25076,29493,1072,1040,12906,1102,2209,1329,1375,4426,3324,11009,2274,1118,3508,1694,1061,25076,29493,2427,1274,6970,4452,22199,8786,1465,1070,1420,2197,1192,11443,29474,29491,1183,13200,1761,1115,1164,17236,1070,1040,13200,1070,1040,18395,1521,7839,2650,1081,19520,1067,10181,1034,29493,1458,7030,1927,1040,7534,6457,2494,1787,6946,29493,7531,1254,9873,781,781,29560,5709,8673,25076,1155,22508,29491,9752,29561,1049,1301,6963,29493,20991,1137,1639,38

In [28]:
import time
import requests

import time
import requests

def gemini_generate(prompt: str, max_retries: int = 5):
    if not GEMINI_URL:
        return "Gemini disabled (missing GEMINI_API_KEY)."

    payload = {"contents": [{"parts": [{"text": prompt}]}]}

    for attempt in range(max_retries):
        r = requests.post(GEMINI_URL, json=payload, timeout=60)

        if r.status_code in (429, 503):
            wait = min(2 ** attempt, 10)  # cap wait to 10s so it doesn't explode
            print(f"⚠️ Gemini busy (status={r.status_code}). Waiting {wait}s then retrying...")
            time.sleep(wait)
            continue

        r.raise_for_status()
        data = r.json()
        return data["candidates"][0]["content"]["parts"][0]["text"]

    return f"Gemini failed after {max_retries} retries (service busy)."


# --- Gemini cache + budget to avoid slow runs ---
GEMINI_CACHE = {}
GEMINI_CALLS = 0

def gemini_cached(prompt: str):
    global GEMINI_CALLS

    if prompt in GEMINI_CACHE:
        return GEMINI_CACHE[prompt]

    if GEMINI_CALLS >= GEMINI_MAX_CALLS:
        return "Gemini skipped: reached GEMINI_MAX_CALLS budget."

    answer = gemini_generate(prompt)
    GEMINI_CACHE[prompt] = answer
    GEMINI_CALLS += 1
    return answer




# 10) RAG Answer Functions

In [29]:
def rag_answer_gemini(query, chunks, index, top_k=5):
    if not USE_GEMINI:
        return "Gemini skipped (USE_GEMINI=False).", []

    retrieved = retrieve(query, chunks, index, top_k=top_k)
    prompt = build_rag_prompt(query, retrieved)
    answer = gemini_cached(prompt)
    return answer, retrieved



# 11) Experiments & Evaluation

In [30]:
import time
import pandas as pd

def timed_call(fn, *args, **kwargs):
    t0 = time.time()
    out = fn(*args, **kwargs)
    t1 = time.time()
    return out, (t1 - t0)


In [31]:
questions = [
    "Which dinosaur is known for having a sail on its back?",
    "Which dinosaur had armor plates on its back?",
    "Which dinosaur was a herbivore with three horns?"
]

def run_experiment(questions, chunks, indexes_dict, top_k=5, local_model="llama3", use_gemini=True):
    rows = []
    for index_name, idx in indexes_dict.items():
        for q in questions:
            (ans_local, retrieved_local), t_local = timed_call(
                rag_answer, q, chunks, idx, top_k=top_k, model=local_model
            )

            if use_gemini and USE_GEMINI:
                try:
                    (ans_ext, retrieved_ext), t_ext = timed_call(
                        rag_answer_gemini, q, chunks, idx, top_k=top_k
                    )
                except Exception as e:
                    ans_ext, t_ext = f"Gemini error: {type(e).__name__}: {e}", None
            else:
                ans_ext, t_ext = "Gemini skipped (disabled for this experiment).", None

            top_sources = ", ".join([r["source"] for r in retrieved_local[:3]])

            rows.append({
                "index": index_name,
                "question": q,
                "top_sources": top_sources,
                "ollama_answer": ans_local,
                "gemini_answer": ans_ext,
                "ollama_time_sec": round(t_local, 3),
                "gemini_time_sec": None if t_ext is None else round(t_ext, 3),
            })
    return pd.DataFrame(rows)



df_results = run_experiment(questions, chunks, INDEXES, top_k=5, local_model="llama3")
df_results


STATUS: 200
RAW TEXT (first 1000 chars):
{"model":"llama3","created_at":"2026-02-01T21:08:03.6892027Z","response":"Spinosaurus.","done":true,"done_reason":"stop","context":[128006,882,128007,271,16533,27785,1701,279,2317,382,53528,512,23956,63989,374,3967,369,3515,264,30503,389,1202,1203,1980,99465,512,58,6540,15570,43613,482,27685,16378,60,577,876,1023,65375,11,32125,5751,44705,43613,11,902,12439,264,2478,3610,1667,6931,304,279,1890,4689,5654,439,3165,15570,43613,11,323,279,23591,356,2171,77140,4987,3778,33254,897,347,3383,75179,43613,11,2643,617,8040,4528,24693,77765,315,872,5309,51313,68,13,578,30503,1253,387,459,24291,315,279,30503,315,279,91821,1122,6925,2690,307,8289,8980,15357,11,902,12439,1603,279,65375,1524,9922,11,9124,555,19873,271,58,6540,15570,43613,482,27685,16378,60,294,68550,11,23377,430,1202,6206,1051,5129,7863,311,1884,315,1023,12903,59076,3447,8032,868,1483,2946,1483,1758,60,15668,16615,30828,993,1572,7982,389,279,1203,5309,51313,68,315,3165,15570,43613,14454,279,819

,index,question,top_sources,ollama_answer,gemini_answer,ollama_time_sec,gemini_time_sec
0,flat,Which dinosaur is known for having a sail on i...,"Spinosaurus - Wikipedia.pdf, Spinosaurus - Wik...",Spinosaurus.,Spinosaurus is known for having a sail on its ...,19.373,1.729
1,flat,Which dinosaur had armor plates on its back?,"Stegosaurus - Wikipedia.pdf, Stegosaurus - Wik...",Stegosaurus.,Stegosaurus,10.670,8.754
2,flat,Which dinosaur was a herbivore with three horns?,"Parasaurolophus - Wikipedia.pdf, Parasauroloph...","Based on the provided context, which only ment...",Gemini failed after 5 retries (service busy).,22.631,25.990
3,hnsw,Which dinosaur is known for having a sail on i...,"Spinosaurus - Wikipedia.pdf, Spinosaurus - Wik...",Spinosaurus.,Spinosaurus is known for having a sail on its ...,11.657,0.034
4,hnsw,Which dinosaur had armor plates on its back?,"Stegosaurus - Wikipedia.pdf, Stegosaurus - Wik...",Stegosaurus.,Stegosaurus,11.190,0.030
5,hnsw,Which dinosaur was a herbivore with three horns?,"Parasaurolophus - Wikipedia.pdf, Parasauroloph...","Based on the provided context, there is no din...",Gemini failed after 5 retries (service busy).,56.652,0.047
6,ivf,Which dinosaur is known for having a sail on i...,"Spinosaurus - Wikipedia.pdf, Spinosaurus - Wik...","According to the context, the dinosaur that is...",Gemini failed after 5 retries (service busy).,30.373,26.465
7,ivf,Which dinosaur had armor plates on its back?,"Stegosaurus - Wikipedia.pdf, Stegosaurus - Wik...",Ankylosaurus.,Gemini failed after 5 retries (service busy).,10.846,26.051
8,ivf,Which dinosaur was a herbivore with three horns?,"Parasaurolophus - Wikipedia.pdf, Parasauroloph...","Based on the context, there is no mention of a...",Gemini failed after 5 retries (service busy).,32.293,26.247


In [32]:
questions = [
    "Which dinosaur is known for having a sail on its back?",
    "Which dinosaur had armor plates on its back?",
    "Which dinosaur was a herbivore with three horns?"
]

def compare_llms(questions, chunks, index, top_k=5, local_models=("llama3", "mistral")):
    results = []
    for q in questions:
        # External (Gemini) once
        ans_ext, src_ext = rag_answer_gemini(q, chunks, index, top_k=top_k)

        row = {
            "question": q,
            "gemini_answer": ans_ext
        }

        # Local models (Ollama): llama3 + mistral
        for m in local_models:
            ans_local, src_local = rag_answer(q, chunks, index, top_k=top_k, model=m)
            row[f"{m}_answer"] = ans_local

            # take sources from the first local model only (or you can store per model)
            if "top_sources" not in row:
                row["top_sources"] = ", ".join(sorted({r["source"] for r in src_local[:3]}))

        results.append(row)

    return results


# choose which index algorithm you want to test here:
index_to_test = INDEXES["hnsw"]   # or INDEXES["flat"] or INDEXES["ivf"]

results = compare_llms(questions, chunks, index_to_test, top_k=5, local_models=("llama3", "mistral"))

for r in results:
    print("\n==============================")
    print("Q:", r["question"])
    print("Top sources:", r["top_sources"])

    print("\nLOCAL (llama3):\n", r["llama3_answer"][:350])
    print("\nLOCAL (mistral):\n", r["mistral_answer"][:350])
    print("\nEXTERNAL (Gemini):\n", r["gemini_answer"][:350])

STATUS: 200
RAW TEXT (first 1000 chars):
{"model":"llama3","created_at":"2026-02-01T21:13:16.9814475Z","response":"Spinosaurus.","done":true,"done_reason":"stop","context":[128006,882,128007,271,16533,27785,1701,279,2317,382,53528,512,23956,63989,374,3967,369,3515,264,30503,389,1202,1203,1980,99465,512,58,6540,15570,43613,482,27685,16378,60,577,876,1023,65375,11,32125,5751,44705,43613,11,902,12439,264,2478,3610,1667,6931,304,279,1890,4689,5654,439,3165,15570,43613,11,323,279,23591,356,2171,77140,4987,3778,33254,897,347,3383,75179,43613,11,2643,617,8040,4528,24693,77765,315,872,5309,51313,68,13,578,30503,1253,387,459,24291,315,279,30503,315,279,91821,1122,6925,2690,307,8289,8980,15357,11,902,12439,1603,279,65375,1524,9922,11,9124,555,19873,271,58,6540,15570,43613,482,27685,16378,60,294,68550,11,23377,430,1202,6206,1051,5129,7863,311,1884,315,1023,12903,59076,3447,8032,868,1483,2946,1483,1758,60,15668,16615,30828,993,1572,7982,389,279,1203,5309,51313,68,315,3165,15570,43613,14454,279,819